In [1]:
# ============================================================
# Download NDBC bulk stdmet data and directional spectral data
# for a user-provided buoy list CSV.
#
# INPUT CSV expected columns (no header required, but header is OK):
#   station_id, station_name, longitude, latitude, water_depth_m,
#   has_directional_data, has_spectra_data
#
# Example row:
# 42012,"NDBC 42012 ...",-87.548,30.06,23.5,Y,Y
#
# OUTPUTS:
#   ndbc_<station>.csv                      # per-buoy stdmet
#   ndbc_bulk_selected_buoys.csv           # combined stdmet
#   ndbc_bulk_selected_buoys.nc            # combined stdmet
#   ndbc_spec_<station>.nc                 # one file per spectral buoy
# ============================================================

import io
import gzip
import requests
import numpy as np
import pandas as pd
import xarray as xr
from pathlib import Path

# ============================================================
# USER SETTINGS
# ============================================================
DATA_DIR = Path("F:/crs/proj/2025_NOPP_comparison/helene_waves")
BUOY_LIST_CSV = DATA_DIR / "ndbc_buoy_list.csv"

START = "2024-09-24"
END   = "2024-09-29"
YEAR  = 2024

OUT_BULK_CSV = DATA_DIR / "ndbc_bulk_selected_buoys.csv"
OUT_BULK_NC  = DATA_DIR / "ndbc_bulk_selected_buoys.nc"
OUT_SUMMARY_CSV = DATA_DIR / "ndbc_selected_buoys_summary.csv"

# NDBC archive layout
STD_MET_URL = "https://www.ndbc.noaa.gov/data/historical/stdmet/{station}h{year}.txt.gz"
SPEC_URLS = {
    "swden":  "https://www.ndbc.noaa.gov/data/historical/swden/{station}w{year}.txt.gz",
    "swdir":  "https://www.ndbc.noaa.gov/data/historical/swdir/{station}d{year}.txt.gz",   # alpha1
    "swdir2": "https://www.ndbc.noaa.gov/data/historical/swdir2/{station}i{year}.txt.gz",  # alpha2
    "swr1":   "https://www.ndbc.noaa.gov/data/historical/swr1/{station}j{year}.txt.gz",    # r1
    "swr2":   "https://www.ndbc.noaa.gov/data/historical/swr2/{station}k{year}.txt.gz",    # r2
}

# ============================================================
# SMALL HELPERS
# ============================================================
def yesno_to_bool(x):
    s = str(x).strip().upper()
    return s in {"Y", "YES", "TRUE", "1"}


def subset_date_range_inclusive(df, start, end):
    if len(df) == 0:
        return df.copy()

    out = df.copy()
    out["time"] = pd.to_datetime(out["time"], utc=True)

    t0 = pd.Timestamp(start, tz="UTC")
    t1 = pd.Timestamp(end, tz="UTC") + pd.Timedelta(days=1) - pd.Timedelta(seconds=1)

    out = out[(out["time"] >= t0) & (out["time"] <= t1)].copy()
    return out.sort_values("time").reset_index(drop=True)


def parse_ndbc_tabular_text(text, station):
    """
    Parse NDBC historical text tables with:
      - header line
      - optional units line
      - whitespace-delimited data
    """
    lines = [ln.rstrip() for ln in text.splitlines() if ln.strip()]
    if len(lines) < 2:
        raise ValueError(f"{station}: file is empty or malformed")

    header_line = lines[0].lstrip("#").strip()
    cols = header_line.split()

    data_start = 1
    if len(lines) > 1:
        second = lines[1].strip()
        toks = second.split()
        if toks and not toks[0].replace("-", "").isdigit():
            data_start = 2

    data_text = "\n".join(lines[data_start:])
    if not data_text.strip():
        raise ValueError(f"{station}: no data rows found")

    df = pd.read_csv(
        io.StringIO(data_text),
        sep=r"\s+",
        names=cols,
        na_values=["MM", "999", "999.0", "9999", "9999.0", "99.00", "999.00"],
        engine="python",
    )

    if {"YY", "MM", "DD", "hh", "mm"}.issubset(df.columns):
        yy = pd.to_numeric(df["YY"], errors="coerce")
        year_vals = np.where(yy < 100, yy + 1900, yy)
        df["time"] = pd.to_datetime(
            {
                "year": year_vals,
                "month": pd.to_numeric(df["MM"], errors="coerce"),
                "day": pd.to_numeric(df["DD"], errors="coerce"),
                "hour": pd.to_numeric(df["hh"], errors="coerce"),
                "minute": pd.to_numeric(df["mm"], errors="coerce"),
            },
            errors="coerce",
            utc=True,
        )
    elif {"YYYY", "MM", "DD", "hh", "mm"}.issubset(df.columns):
        df["time"] = pd.to_datetime(
            {
                "year": pd.to_numeric(df["YYYY"], errors="coerce"),
                "month": pd.to_numeric(df["MM"], errors="coerce"),
                "day": pd.to_numeric(df["DD"], errors="coerce"),
                "hour": pd.to_numeric(df["hh"], errors="coerce"),
                "minute": pd.to_numeric(df["mm"], errors="coerce"),
            },
            errors="coerce",
            utc=True,
        )
    else:
        raise ValueError(f"{station}: could not identify time columns")

    return df


def download_gz_text(url):
    r = requests.get(url, timeout=120)
    if r.status_code == 404:
        raise FileNotFoundError(url)
    r.raise_for_status()
    with gzip.GzipFile(fileobj=io.BytesIO(r.content)) as gz:
        return gz.read().decode("utf-8", errors="replace")


# ============================================================
# BULK STDMET DOWNLOAD
# ============================================================
def download_ndbc_stdmet_station_year(station, year=2024):
    url = STD_MET_URL.format(station=station, year=year)
    text = download_gz_text(url)
    df = parse_ndbc_tabular_text(text, station)

    for c in [
        "WDIR", "WSPD", "GST", "WVHT", "DPD", "APD", "MWD",
        "PRES", "ATMP", "WTMP", "DEWP", "VIS", "PTDY", "TIDE"
    ]:
        if c in df.columns:
            df[c] = pd.to_numeric(df[c], errors="coerce")

    df["station"] = str(station)
    return df


def download_ndbc_bulk_stats(buoys, start, end, year=2024):
    """
    Re-used bulk downloader.
    """
    per_buoy = {}

    for buoy in buoys:
        try:
            df = download_ndbc_stdmet_station_year(buoy, year=year)
            df = subset_date_range_inclusive(df, start=start, end=end)
            per_buoy[buoy] = df
            print(f"{buoy}: {len(df)} bulk records")
        except Exception as e:
            print(f"{buoy}: bulk FAILED -> {e}")
            per_buoy[buoy] = pd.DataFrame()

    combined = (
        pd.concat(
            [df for df in per_buoy.values() if len(df) > 0],
            ignore_index=True,
            sort=False,
        )
        if any(len(df) > 0 for df in per_buoy.values())
        else pd.DataFrame()
    )

    if len(combined) > 0:
        preferred_cols = [
            "station", "time",
            "WDIR", "WSPD", "GST", "WVHT", "DPD", "APD", "MWD",
            "PRES", "ATMP", "WTMP", "DEWP", "VIS", "PTDY", "TIDE",
        ]
        cols = [c for c in preferred_cols if c in combined.columns] + [c for c in combined.columns if c not in preferred_cols]
        combined = combined[cols].sort_values(["station", "time"]).reset_index(drop=True)

    return combined, per_buoy


# ============================================================
# SAVE BULK TO NETCDF
# ============================================================
def build_ndbc_xarray(df, metadata, buoys):
    """
    Convert combined NDBC dataframe to xarray Dataset with dims:
      time, station
    """
    if len(df) == 0:
        raise ValueError("Input dataframe is empty")

    df = df.copy()
    metadata = metadata.copy()

    df["time"] = pd.to_datetime(df["time"], utc=True).dt.tz_convert(None)

    time = np.sort(df["time"].dropna().unique()).astype("datetime64[ns]")
    stations = [b for b in buoys if b in df["station"].unique()]
    ntime = len(time)
    nsta = len(stations)

    meta = (
        metadata.drop_duplicates(subset="station")
        .set_index("station")
        .reindex(stations)
    )

    ds = xr.Dataset(
        coords={
            "time": ("time", time),
            "station": np.arange(nsta, dtype=np.int32),
        }
    )

    ds["station_id"] = ("station", np.asarray(stations, dtype="S8"))

    if "station_name" in meta.columns:
        ds["station_name"] = ("station", meta["station_name"].fillna("").astype("S128"))

    for name, units in [
        ("latitude", "degrees_north"),
        ("longitude", "degrees_east"),
        ("site_elevation_m", "m"),
        ("water_depth_m", "m"),
    ]:
        if name in meta.columns:
            ds[name] = ("station", meta[name].to_numpy(dtype=np.float32))
            ds[name].attrs["units"] = units

    var_map = {
        "WVHT": ("wave_height", "m"),
        "DPD":  ("dominant_period", "s"),
        "APD":  ("average_period", "s"),
        "MWD":  ("mean_wave_direction", "deg"),
        "WDIR": ("wind_direction", "deg"),
        "WSPD": ("wind_speed", "m s-1"),
        "GST":  ("wind_gust", "m s-1"),
        "PRES": ("air_pressure", "hPa"),
        "ATMP": ("air_temperature", "degC"),
        "WTMP": ("water_temperature", "degC"),
    }

    time_index = pd.Index(time)
    station_index = {sta: i for i, sta in enumerate(stations)}

    for src_name, (out_name, units) in var_map.items():
        arr = np.full((ntime, nsta), np.nan, dtype=np.float32)

        if src_name in df.columns:
            for sta, g in df.groupby("station"):
                if sta not in station_index:
                    continue

                j = station_index[sta]
                gg = (
                    g[["time", src_name]]
                    .dropna()
                    .drop_duplicates(subset="time")
                    .sort_values("time")
                )

                if len(gg) == 0:
                    continue

                it = time_index.get_indexer(gg["time"].to_numpy().astype("datetime64[ns]"))
                good = it >= 0
                arr[it[good], j] = gg[src_name].to_numpy(dtype=np.float32)[good]

        ds[out_name] = (("time", "station"), arr)
        ds[out_name].attrs["units"] = units
        ds[out_name].attrs["source_name"] = src_name

    return ds


def save_ndbc_to_netcdf(df, metadata, buoys, out_nc):
    ds = build_ndbc_xarray(df, metadata, buoys)
    try:
        ds.to_netcdf(out_nc, engine="netcdf4")
    except Exception:
        ds.to_netcdf(out_nc)
    return ds


# ============================================================
# DIRECTIONAL SPECTRA DOWNLOAD
# ============================================================
def download_ndbc_spectral_component(station, year, component):
    """
    component in: swden, swdir, swdir2, swr1, swr2
    """
    url = SPEC_URLS[component].format(station=station, year=year)
    text = download_gz_text(url)
    df = parse_ndbc_tabular_text(text, station)

    # Convert all non-time/date columns to numeric
    non_num = {"YY", "YYYY", "MM", "DD", "hh", "mm", "time"}
    for c in df.columns:
        if c not in non_num:
            df[c] = pd.to_numeric(df[c], errors="coerce")

    # NDBC says historical r1/r2 are scaled by 100
    if component in {"swr1", "swr2"}:
        data_cols = [c for c in df.columns if c not in non_num]
        df[data_cols] = df[data_cols] * 0.01

    df = subset_date_range_inclusive(df, START, END)
    return df


def infer_frequency_columns(df):
    """
    NDBC spectral files store frequencies as column names after the date/time columns.
    """
    exclude = {"YY", "YYYY", "MM", "DD", "hh", "mm", "time"}
    freq_cols = [c for c in df.columns if c not in exclude]
    freqs = np.array([float(c) for c in freq_cols], dtype=float)
    return freq_cols, freqs


def build_spectral_dataset_for_buoy(station, meta_row, bulk_df, year=2024):
    """
    Download five directional spectral components for one buoy and build one Dataset.
    """
    comp_map = {
        "swden":  "spectral_wave_density",
        "swdir":  "alpha1",
        "swdir2": "alpha2",
        "swr1":   "r1",
        "swr2":   "r2",
    }

    comp_dfs = {}
    for comp in comp_map:
        comp_dfs[comp] = download_ndbc_spectral_component(station, year=year, component=comp)
        print(f"{station}: {comp} records = {len(comp_dfs[comp])}")

    # Require at least swden to proceed
    if len(comp_dfs["swden"]) == 0:
        raise ValueError(f"{station}: no spectral density records in requested window")

    freq_cols, freqs = infer_frequency_columns(comp_dfs["swden"])
    time = pd.to_datetime(comp_dfs["swden"]["time"], utc=True).dt.tz_convert(None).to_numpy()

    ds = xr.Dataset(
        coords={
            "time": ("time", time),
            "frequency": ("frequency", freqs.astype(np.float32)),
        }
    )

    # Station metadata
    ds["station_id"] = xr.DataArray(np.asarray(station, dtype="S8"))
    ds["station_name"] = xr.DataArray(np.asarray(str(meta_row["station_name"]), dtype="S128"))
    ds["longitude"] = xr.DataArray(np.float32(meta_row["longitude"]))
    ds["latitude"] = xr.DataArray(np.float32(meta_row["latitude"]))
    ds["water_depth_m"] = xr.DataArray(np.float32(meta_row["water_depth_m"]))

    ds["longitude"].attrs["units"] = "degrees_east"
    ds["latitude"].attrs["units"] = "degrees_north"
    ds["water_depth_m"].attrs["units"] = "m"

    # Spectral variables
    units_map = {
        "swden": "m^2 Hz^-1",
        "swdir": "deg",
        "swdir2": "deg",
        "swr1": "1",
        "swr2": "1",
    }

    for comp, out_name in comp_map.items():
        dfi = comp_dfs[comp].copy()
        if len(dfi) == 0:
            arr = np.full((len(time), len(freq_cols)), np.nan, dtype=np.float32)
        else:
            dfi["time"] = pd.to_datetime(dfi["time"], utc=True).dt.tz_convert(None)
            dfi = dfi.set_index("time").reindex(pd.DatetimeIndex(time))
            arr = dfi[freq_cols].to_numpy(dtype=np.float32)

        ds[out_name] = (("time", "frequency"), arr)
        ds[out_name].attrs["units"] = units_map[comp]
        ds[out_name].attrs["source_component"] = comp

    # Add bulk stats on the same time axis
    if bulk_df is not None and len(bulk_df) > 0:
        b = bulk_df.copy()
        b["time"] = pd.to_datetime(b["time"], utc=True).dt.tz_convert(None)
        b = b.set_index("time").sort_index()

        bulk_map = {
            "WVHT": ("wave_height", "m"),
            "DPD": ("dominant_period", "s"),
            "APD": ("average_period", "s"),
            "MWD": ("mean_wave_direction", "deg"),
            "WDIR": ("wind_direction", "deg"),
            "WSPD": ("wind_speed", "m s-1"),
            "GST": ("wind_gust", "m s-1"),
            "PRES": ("air_pressure", "hPa"),
            "ATMP": ("air_temperature", "degC"),
            "WTMP": ("water_temperature", "degC"),
        }

        # nearest within 30 min
        b = b.reindex(pd.DatetimeIndex(time), method="nearest", tolerance=pd.Timedelta("30min"))

        for src_name, (out_name, units) in bulk_map.items():
            if src_name in b.columns:
                ds[out_name] = (("time",), b[src_name].to_numpy(dtype=np.float32))
                ds[out_name].attrs["units"] = units
                ds[out_name].attrs["source_name"] = src_name

    return ds


# ============================================================
# READ BUOY LIST
# ============================================================
def read_buoy_list_csv(path):
    """
    Read the buoy list CSV. Works with or without a header row.
    """
    df0 = pd.read_csv(path)

    lower_cols = [str(c).strip().lower() for c in df0.columns]
    expected = ["station_id", "station_name", "longitude", "latitude", "water_depth_m"]

    if all(c in lower_cols for c in expected):
        rename_map = {old: new for old, new in zip(df0.columns, lower_cols)}
        df = df0.rename(columns=rename_map).copy()
    else:
        df = pd.read_csv(
            path,
            header=None,
            names=[
                "station_id",
                "station_name",
                "longitude",
                "latitude",
                "water_depth_m",
                "has_directional_data",
                "has_spectra_data",
            ],
        )

    df["station_id"] = df["station_id"].astype(str).str.strip()
    df["longitude"] = pd.to_numeric(df["longitude"], errors="coerce")
    df["latitude"] = pd.to_numeric(df["latitude"], errors="coerce")
    df["water_depth_m"] = pd.to_numeric(df["water_depth_m"], errors="coerce")

    if "has_directional_data" in df.columns:
        df["has_directional_data"] = df["has_directional_data"].map(yesno_to_bool)
    else:
        df["has_directional_data"] = False

    if "has_spectra_data" in df.columns:
        df["has_spectra_data"] = df["has_spectra_data"].map(yesno_to_bool)
    else:
        df["has_spectra_data"] = False

    df = df.rename(columns={"station_id": "station"})
    return df


# ============================================================
# RUN
# ============================================================
meta = read_buoy_list_csv(BUOY_LIST_CSV)
buoys = meta["station"].tolist()

print("Buoys to process:")
print(meta[["station", "station_name", "has_directional_data", "has_spectra_data"]])

# ------------------------------------------------------------
# 1) BULK DATA
# ------------------------------------------------------------
df_bulk, per_buoy_bulk = download_ndbc_bulk_stats(
    buoys=buoys,
    start=START,
    end=END,
    year=YEAR,
)

# save per-buoy bulk csv
for sta, df_sta in per_buoy_bulk.items():
    if len(df_sta) == 0:
        continue

    row = meta.loc[meta["station"] == sta].iloc[0]
    out = df_sta.copy()
    out["station_name"] = row["station_name"]
    out["longitude"] = row["longitude"]
    out["latitude"] = row["latitude"]
    out["water_depth_m"] = row["water_depth_m"]

    out_csv = DATA_DIR / f"ndbc_{sta}.csv"
    out.to_csv(out_csv, index=False)
    print(f"Saved bulk CSV: {out_csv}")

# combined bulk outputs
if len(df_bulk) > 0:
    df_bulk.to_csv(OUT_BULK_CSV, index=False)
    print(f"Saved combined bulk CSV: {OUT_BULK_CSV}")

    ds_bulk = save_ndbc_to_netcdf(
        df=df_bulk,
        metadata=meta,
        buoys=buoys,
        out_nc=OUT_BULK_NC,
    )
    print(f"Saved combined bulk NetCDF: {OUT_BULK_NC}")
else:
    ds_bulk = None
    print("No bulk data downloaded.")

# ------------------------------------------------------------
# 2) SUMMARY TABLE
# ------------------------------------------------------------
summary_rows = []
for _, row in meta.iterrows():
    sta = row["station"]
    df_sta = per_buoy_bulk.get(sta, pd.DataFrame())

    summary_rows.append({
        "station": sta,
        "station_name": row["station_name"],
        "longitude": row["longitude"],
        "latitude": row["latitude"],
        "water_depth_m": row["water_depth_m"],
        "has_directional_data_flag": bool(row["has_directional_data"]),
        "has_spectra_data_flag": bool(row["has_spectra_data"]),
        "n_bulk_records": len(df_sta),
        "has_wave_height": ("WVHT" in df_sta.columns and df_sta["WVHT"].notna().any()) if len(df_sta) else False,
        "has_dominant_period": ("DPD" in df_sta.columns and df_sta["DPD"].notna().any()) if len(df_sta) else False,
        "has_average_period": ("APD" in df_sta.columns and df_sta["APD"].notna().any()) if len(df_sta) else False,
        "has_mean_wave_direction": ("MWD" in df_sta.columns and df_sta["MWD"].notna().any()) if len(df_sta) else False,
        "max_wave_height_m": df_sta["WVHT"].max(skipna=True) if len(df_sta) and "WVHT" in df_sta.columns else np.nan,
    })

summary_df = pd.DataFrame(summary_rows)
summary_df.to_csv(OUT_SUMMARY_CSV, index=False)
print(f"Saved summary CSV: {OUT_SUMMARY_CSV}")

# ------------------------------------------------------------
# 3) DIRECTIONAL SPECTRA FILES
# ------------------------------------------------------------
spec_rows = meta.loc[meta["has_spectra_data"]].copy()

for _, row in spec_rows.iterrows():
    sta = row["station"]
    try:
        bulk_sta = per_buoy_bulk.get(sta, pd.DataFrame())
        ds_spec = build_spectral_dataset_for_buoy(
            station=sta,
            meta_row=row,
            bulk_df=bulk_sta,
            year=YEAR,
        )

        out_nc = DATA_DIR / f"ndbc_spec_{sta}.nc"
        try:
            ds_spec.to_netcdf(out_nc, engine="netcdf4")
        except Exception:
            ds_spec.to_netcdf(out_nc)

        print(f"Saved spectral NetCDF: {out_nc}")

    except Exception as e:
        print(f"{sta}: spectral FAILED -> {e}")

Buoys to process:
  station                                       station_name  \
0    name                                           longname   
1   42012  NDBC 42012 (LLNR 138) Orange Beach - 44 nm SE ...   
2   42036  NDBC 42036 (LLNR 855)- West Tampa - 112 nm WNW...   
3   42039  NDBC 42039 (LLNR 115) - Pensacola - 115 nm SSE...   
4   42040  NDBC 42040 (LLNR 245) - Luke Offshore Test Pla...   
5   42095                 NDBC 42095 - Satan Shoal, FL (244)   
6   42097                          NDBC 42097 - Pulley Ridge   
7   42098               NDBC 42098 - Egmont Channel Entrance   

   has_directional_data  has_spectra_data  
0                 False             False  
1                  True              True  
2                  True              True  
3                  True              True  
4                  True              True  
5                 False             False  
6                 False             False  
7                  True             False  
name: bul